In [ ]:
# --- IMPORTS ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuraciones visuales
plt.style.use('seaborn')
sns.set_palette('viridis')
pd.set_option("display.max_colwidth", 200)

print("Librerías cargadas correctamente.")

In [ ]:
# --- LOAD DATASET ---

file_path = "../data/tickets_train.csv"  # ajusta si tu notebook está en otra carpeta

df = pd.read_csv(file_path)
df.head()

In [ ]:
# --- BASIC INFO ---
print("Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
#Estadísticas Generales
df.describe(include='all')

In [ ]:
#Distribución del tipo de ticket
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='ticket_type')
plt.title("Distribución: Tipo de Ticket (Correctivo vs Evolutivo)")
plt.xlabel("Tipo de Ticket")
plt.ylabel("Cantidad")
plt.show()


In [ ]:
#Distribución del Riesgo de Churn
plt.figure(figsize=(8,4))
sns.histplot(df['churn_risk'], bins=20, kde=True)
plt.title("Distribución del Riesgo de Churn (0–100)")
plt.xlabel("Churn Risk")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
#Relación entre tipo de ticket y churn
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x='ticket_type', y='churn_risk')
plt.title("Churn Risk por Tipo de Ticket")
plt.xlabel("Tipo de Ticket")
plt.ylabel("Churn Risk")
plt.show()


In [ ]:
#Análisis de Sentimiento
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='sentiment_label')
plt.title("Distribución del Sentimiento")
plt.xlabel("Sentimiento")
plt.ylabel("Cantidad")
plt.show()


In [ ]:
#¿El phishing impacta el churn?
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x='is_phishing', y='churn_risk')
plt.title("Churn Risk según presencia de phishing")
plt.xlabel("Phishing (0 = No, 1 = Sí)")
plt.ylabel("Churn Risk")
plt.show()


In [ ]:
#Correlaciones entre variables numéricas
# Seleccionar solo columnas numéricas
numeric_df = df.select_dtypes(include=[np.number])

plt.figure(figsize=(10,6))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de Correlación")
plt.show()


In [ ]:
#Top Drivers de Churn
correlations = numeric_df.corr()['churn_risk'].sort_values(ascending=False)

print("Top variables correlacionadas con churn (mayor a menor):")
correlations


In [ ]:
#Guardar drivers de churn para usar en el dashboard
drivers = correlations.to_frame().reset_index()
drivers.columns = ['feature', 'correlation_with_churn']

drivers.to_csv("../models/churn_drivers.csv", index=False)

print("Drivers guardados en models/churn_drivers.csv")


In [ ]:
#Vista rápida de tickets extremadamente críticos
df[df['churn_risk'] > 80].head(10)


In [ ]:
from IPython.display import Markdown as md

md("""
#  Conclusiones del EDA

**1. Tipo de ticket**
- Los tickets Correctivos suelen ser más frecuentes.
- Los tickets Correctivos tienen **churn más alto** que los Evolutivos.

**2. Sentimiento**
- Los tickets negativos están asociados con mayor churn.

**3. Phishing**
- Los tickets con phishing muestran mayor riesgo promedio.

**4. Drivers**
- Las variables más correlacionadas con churn son:
  - Número de tickets correctivos recientes.
  - Sentimiento negativo.
  - Presencia de phishing.
  - Edad del proyecto.

**5. Insight general**
→ *Si un cliente acumula muchos tickets correctivos + sentimiento negativo, el churn se dispara.*
""")
